In [ ]:
#| hide
from compact.core import *
from compact.types import Symbol
from compact import lisp
from fasthtml.common import *

In [ ]:
lisp.register_magic()

# Example blog

> Implement a blog using embedded lisp in python with fasthtml

In [ ]:
app, rt = fast_app(hdrs=[Style("""
    body      { max-width: 740px; margin: 3em auto; font-family: serif; line-height: 1.6; }
    pre       { background: #f5f5f5; padding: 1em; overflow-x: auto; border-left: 3px solid #ccc; }
    .meta     { color: #999; font-size: 0.85em; }
    .result   { color: #2a2; font-style: italic; }
    .example  { background: #f9f9f9; padding: 0.75em 1em; margin: 1em 0; }
    .footnote { border-top: 1px solid #eee; font-size: 0.85em; color: #666; margin-top: 2em; }
    .series   { color: #66a; }
""")])



In [ ]:
def kwargs(pairs): return {k.s if isinstance(k, Symbol) else k:v for k,v in pairs}

In [ ]:
def call(fn, args):
    pos = [a[0] for a in args if len(a) == 1]
    kw  = {k.s: v for k, v in (a for a in args if len(a) == 2)}
    return fn(*pos, **kw)

In [ ]:
"""
(call P `(("hello") (cls "p-5")))
""" @ lisp

<p class="p-5">hello</p>

In [ ]:
"""
(define (caar  p) (car (car p)))
(define (cadr  p) (car (cdr p)))
(define (cdar  p) (cdr (car p)))
(define (cddr  p) (cdr (cdr p)))
(define (caddr  p) (car (cdr (cdr p))))
(define (cadddr p) (car (cdr (cdr (cdr p)))))
(define (cddddr p) (cdr (cdr (cdr (cdr p)))))
""" @ lisp

'cddddr'

In [ ]:
"""
(define (make-post slug title date . body)
  (list 'post slug title date body))

(define (post? p)      (and (pair? p) (equal? (car p) 'post)))
(define (post-slug p)  (cadr p))
(define (post-title p) (caddr p))
(define (post-date p)  (cadddr p))
(define (post-body p)  (car (cddddr p)))
""" @ lisp

'post-body'

In [ ]:
"""
(define (fact n)
  (if (= n 0) 1
      (* n (fact (- n 1)))))

(make-post "recursion" "On Recursion" "2024-01-10"
  (list 'prose
    "Recursion is defining something in terms of itself. "
    "A procedure calls itself on a smaller input "
    "until it reaches a base case.")
  (list 'code-block
    "(define (fact n)"
    "  (if (= n 0) 1"
    "      (* n (fact (- n 1)))))")
  (list 'example "(fact 10)" (fact 10))
  (list 'footnote
    "This generates a recursive process — the chain of deferred "
    "multiplications grows linearly with n. An iterative version "
    "using an accumulator would not."))
""" @ lisp

[compact.types.Symbol(s='post'),
 'recursion',
 'On Recursion',
 '2024-01-10',
 [[compact.types.Symbol(s='prose'),
   'Recursion is defining something in terms of itself. ',
   'A procedure calls itself on a smaller input ',
   'until it reaches a base case.'],
  [compact.types.Symbol(s='code-block'),
   '(define (fact n)',
   '  (if (= n 0) 1',
   '      (* n (fact (- n 1)))))'],
  [compact.types.Symbol(s='example'), '(fact 10)', 3628800],
  [compact.types.Symbol(s='footnote'),
   'This generates a recursive process — the chain of deferred ',
   'multiplications grows linearly with n. An iterative version ',
   'using an accumulator would not.']]]

In [ ]:
"""
(define (my-length lst)
  (define (iter l acc)
    (if (null? l) acc
        (iter (cdr l) (+ acc 1))))
  (iter lst 0))

(make-post "lists" "On Lists" "2024-01-17"
  (list 'prose
    "A list is either empty, or a value consed onto another list. "
    "This definition is recursive, so many list operations are too.")
  (list 'code-block
    "(define (my-length lst)"
    "  (define (iter l acc)"
    "    (if (null? l) acc"
    "        (iter (cdr l) (+ acc 1))))"
    "  (iter lst 0))")
  (list 'example "(my-length '(a b c d e))" (my-length '(a b c d e)))
  (list 'footnote
    "my-length generates an iterative process despite being written "
    "recursively. The accumulator carries the state; "
    "the interpreter need not grow the stack."))
""" @ lisp

[compact.types.Symbol(s='post'),
 'lists',
 'On Lists',
 '2024-01-17',
 [[compact.types.Symbol(s='prose'),
   'A list is either empty, or a value consed onto another list. ',
   'This definition is recursive, so many list operations are too.'],
  [compact.types.Symbol(s='code-block'),
   '(define (my-length lst)',
   '  (define (iter l acc)',
   '    (if (null? l) acc',
   '        (iter (cdr l) (+ acc 1))))',
   '  (iter lst 0))'],
  [compact.types.Symbol(s='example'), "(my-length '(a b c d e))", 5],
  [compact.types.Symbol(s='footnote'),
   'my-length generates an iterative process despite being written ',
   'recursively. The accumulator carries the state; ',
   'the interpreter need not grow the stack.']]]

In [ ]:
"""
(define (render-node node)
  (let ((tag (car node))
        (content (cdr node)))
    (cond
      ((equal? tag 'prose)      (apply P content))
      ((equal? tag 'code-block) (Pre (apply Code content)))
      ((equal? tag 'example)    (Div (Code (car content))
                                     (Span (string-append " → " (number->string (cadr content))))))
      ((equal? tag 'footnote)   (apply Aside content))
      (else (error "unknown node type" tag)))))

(define (render-post p)
  (apply Main
    (H1 (post-title p))
    (P (post-date p))
    (map render-node (post-body p))))
""" @ lisp

'render-post'

In [ ]:
"""
(render-post (make-post "recursion" "On Recursion" "2024-01-10"
  (list 'prose "Recursion is defining something in terms of itself.")
  (list 'example "(fact 10)" (fact 10))))
""" @ lisp

<main><h1>On Recursion</h1><p>2024-01-10</p><p>Recursion is defining something in terms of itself.</p><div><code>(fact 10)</code><span> → 3628800</span></div></main>

In [ ]:
"""
(render-post (make-post "recursion" "On Recursion" "2024-01-10"
  (list 'prose
    "Recursion is defining something in terms of itself. "
    "A procedure calls itself on a smaller input "
    "until it reaches a base case.")
  (list 'code-block
    "(define (fact n)"
    "  (if (= n 0) 1"
    "      (* n (fact (- n 1)))))")
  (list 'example "(fact 10)" (fact 10))
  (list 'footnote
    "This generates a recursive process — the chain of deferred "
    "multiplications grows linearly with n. An iterative version "
    "using an accumulator would not.")))
""" @ lisp

<main><h1>On Recursion</h1><p>2024-01-10</p><p>Recursion is defining something in terms of itself. A procedure calls itself on a smaller input until it reaches a base case.</p><pre><code>(define (fact n)  (if (= n 0) 1      (* n (fact (- n 1)))))</code></pre><div><code>(fact 10)</code><span> → 3628800</span></div><aside>This generates a recursive process — the chain of deferred multiplications grows linearly with n. An iterative version using an accumulator would not.</aside></main>

In [ ]:
"""
(define *posts* '())

(define (register-post p)
  (set! *posts* (cons p *posts*)))

(define (find-post slug)
  (define (search ps)
    (cond ((null? ps) #f)
          ((equal? (post-slug (car ps)) slug) (car ps))
          (else (search (cdr ps)))))
  (search *posts*))

(define (all-posts) (reverse *posts*))
""" @ lisp

'all-posts'

In [ ]:
"""
(register-post (make-post "recursion" "On Recursion" "2024-01-10"
  (list 'prose
    "Recursion is defining something in terms of itself. "
    "A procedure calls itself on a smaller input "
    "until it reaches a base case.")
  (list 'code-block
    "(define (fact n)"
    "  (if (= n 0) 1"
    "      (* n (fact (- n 1)))))")
  (list 'example "(fact 10)" (fact 10))
  (list 'footnote
    "This generates a recursive process — the chain of deferred "
    "multiplications grows linearly with n. An iterative version "
    "using an accumulator would not.")))

(register-post (make-post "lists" "On Lists" "2024-01-17"
  (list 'prose
    "A list is either empty, or a value consed onto another list. "
    "This definition is recursive, so many list operations are too.")
  (list 'code-block
    "(define (my-length lst)"
    "  (define (iter l acc)"
    "    (if (null? l) acc"
    "        (iter (cdr l) (+ acc 1))))"
    "  (iter lst 0))")
  (list 'example "(my-length '(a b c d e))" (my-length '(a b c d e)))
  (list 'footnote
    "my-length generates an iterative process despite being written "
    "recursively. The accumulator carries the state; "
    "the interpreter need not grow the stack.")))
""" @ lisp

'*posts*'

In [ ]:
app, rt = fast_app()

@rt("/root")
def get():
    return Main(H1("A Lisp Blog"),
                Ul(*[Li(A(post_title(p), href=f"/root/post/{post_slug(p)}")) 
                     for p in all_posts()]))

@rt("/root/post/{slug}")
def get(slug: str):
    p = find_post(slug)
    if not p: return P("not found")
    return render_post(p)

serve()

In [ ]:
%%lisp
(call P `(("hello") (cls "p-5")))


<p class="p-5">hello</p>